# Отчет по анализу массовых регистраций юридических лиц
## Fraud-Audit исследование

**Контекст:** Внутренний аудит банка — выявление потенциальных мошеннических паттернов

**Гипотеза:** Компании из массовых кластеров (>17 фирм/координату) в пиковые месяцы санкций — подставные структуры

---


## 1. Настройка окружения


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os

plt.style.use('default')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

print(f"Рабочая директория: {BASE_DIR}")
print(f"Данные: {os.path.exists(DATA_DIR)}")
print(f"Результаты: {os.path.exists(RESULTS_DIR)}")


## 2. Загрузка данных


In [ ]:
df_rfsd = pd.read_parquet(os.path.join(DATA_DIR, "rfsd_2020_2024.parquet"))
print(f"RFSD: {df_rfsd.shape[0]:,} записей, {df_rfsd.shape[1]} колонок")

with open(os.path.join(DATA_DIR, "runsd", "nsd_isin.json"), "r", encoding="utf-8") as f:
    records = [json.loads(line.strip()) for line in f if line.strip()]
df_nsd = pd.DataFrame(records)
print(f"RuNSD: {df_nsd.shape[0]:,} записей")

df_clusters = pd.read_csv(os.path.join(DATA_DIR, "clusters_for_sql.csv"))
print(f"Кластеры: {df_clusters.shape[0]:,} записей")

df_rfsd.head(3)


## 3. Первичный анализ данных (EDA)


In [ ]:
region_stats = df_rfsd['region'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(region_stats)), region_stats.values, color='steelblue', alpha=0.8)
ax.set_yticks(range(len(region_stats)))
ax.set_yticklabels(region_stats.index)
ax.set_xlabel('Количество компаний')
ax.set_title('ТОП-15 регионов по количеству компаний', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for i, (bar, val) in enumerate(zip(bars, region_stats.values)):
    ax.text(val + 100, bar.get_y() + bar.get_height()/2, f'{val:,}',
            va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Всего регионов: {df_rfsd['region'].nunique()}")


## 4. Анализ санкционных данных


In [ ]:
def extract_prop(props, key):
    if isinstance(props, dict) and key in props:
        val = props[key]
        return val[0] if isinstance(val, list) and val else val
    return None

df_nsd['first_seen'] = pd.to_datetime(df_nsd['first_seen'], errors='coerce')
df_nsd['topics'] = df_nsd['properties'].apply(lambda p: extract_prop(p, 'topics'))

def is_sanction(topics):
    if isinstance(topics, list):
        return any('sanction' in str(t).lower() for t in topics)
    return 'sanction' in str(topics).lower()

df_nsd['is_sanction'] = df_nsd['topics'].apply(is_sanction)

df_nsd['year_month'] = df_nsd['first_seen'].dt.to_period('M')
sanction_by_month = df_nsd[df_nsd['is_sanction']].groupby('year_month').size()

fig, ax = plt.subplots(figsize=(14, 5))
sanction_by_month.plot(kind='bar', ax=ax, color='darkred', alpha=0.7)
ax.set_title('Динамика санкционных записей по месяцам', fontsize=14, fontweight='bold')
ax.set_xlabel('Месяц')
ax.set_ylabel('Количество записей')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print(f"Всего санкционных записей: {df_nsd['is_sanction'].sum():,}")


## 5. Анализ кластеров массовой регистрации


In [ ]:
sizes = df_clusters['companies_count'][df_clusters['companies_count'] >= 2]
q1, q3 = sizes.quantile([0.25, 0.75])
iqr = q3 - q1
threshold = q3 + 1.5 * iqr

print('=' * 50)
print('СТАТИСТИКА КЛАСТЕРОВ')
print('=' * 50)
print(f'Всего кластеров: {len(df_clusters):,}')
print(f'Кластеров (≥2 компаний): {len(sizes):,}')
print(f'  Средний размер: {sizes.mean():.1f}')
print(f'  Медиана: {sizes.median():.1f}')
print(f'  Максимум: {sizes.max():,}')
print(f'  Q3: {q3:.0f}')
print(f'\nIQR Порог: {threshold:.0f}')
print(f'Аномалий (> {threshold:.0f}): {(df_clusters["companies_count"] > threshold).sum():,}')
print('=' * 50)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
ax1.hist(sizes, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax1.axvline(threshold, color='darkred', linestyle='--', linewidth=2,
            label=f'IQR порог ({threshold:.0f})')
ax1.set_xlim(0, min(200, sizes.max()))
ax1.set_xlabel('Компаний в кластере')
ax1.set_ylabel('Частота')
ax1.set_title('Распределение размеров кластеров', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
bp = ax2.boxplot(sizes[sizes <= 100], vert=True, patch_artist=True,
                   boxprops=dict(facecolor='lightblue', alpha=0.7),
                   medianprops=dict(color='darkred', linewidth=2))
ax2.set_ylabel('Компаний в кластере')
ax2.set_title('Box plot размеров кластеров (≤100)', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Географическая визуализация


In [ ]:
top10 = df_clusters.nlargest(10, 'companies_count')

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(top10)))
bars = ax.barh(range(len(top10)), top10["companies_count"].values, color=colors)

labels = [f"{row['region'][:20]} ({row['companies_count']})"
          for _, row in top10.iterrows()]
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(labels)
ax.set_xlabel('Количество компаний')
ax.set_title('ТОП-10 массовых адресов регистрации', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for i, (bar, val) in enumerate(zip(bars, top10["companies_count"].values)):
    ax.text(val + 20, bar.get_y() + bar.get_height()/2, f'{val:,}',
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
anomalous = df_clusters[df_clusters['companies_count'] > threshold]
region_anomaly = anomalous['region'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 6))
wedges, texts, autotexts = ax.pie(
    region_anomaly.values,
    labels=region_anomaly.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=plt.cm.Set3(np.linspace(0, 1, len(region_anomaly)))
)
ax.set_title('Распределение аномальных кластеров по регионам',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Всего аномальных кластеров: {len(anomalous)}')


## 7. Heatmap рисков


In [ ]:
def categorize_cluster(count):
    if count == 1: return 'Single (1)'
    elif count <= 5: return 'Small (2-5)'
    elif count <= 20: return 'Medium (6-20)'
    elif count <= 100: return 'Large (21-100)'
    else: return 'Mass (>100)'
df_clusters['size_category'] = df_clusters['companies_count'].apply(categorize_cluster)
pivot = df_clusters.groupby(['region', 'size_category']).size().unstack(fill_value=0)
top_regions = df_clusters['region'].value_counts().head(15).index
pivot_top = pivot.loc[top_regions]
cat_order = ['Single (1)', 'Small (2-5)', 'Medium (6-20)', 'Large (21-100)', 'Mass (>100)']
pivot_top = pivot_top[[c for c in cat_order if c in pivot_top.columns]]

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot_top.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot_top.columns)))
ax.set_yticks(range(len(pivot_top.index)))
ax.set_xticklabels(pivot_top.columns, fontsize=10)
ax.set_yticklabels(pivot_top.index, fontsize=9)

for i in range(len(pivot_top.index)):
    for j in range(len(pivot_top.columns)):
        val = pivot_top.iloc[i, j]
        if val > 0:
            ax.text(j, i, str(int(val)), ha='center', va='center',
                    color='white' if val > pivot_top.values.max() * 0.5 else 'black',
                    fontsize=9)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Количество кластеров', rotation=270, labelpad=15)
ax.set_title('Тепловая карта: Распределение кластеров по регионам и размерам',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Категория размера кластера')
ax.set_ylabel('Регион')
plt.tight_layout()
plt.show()


## 8. Выводы и рекомендации


In [ ]:
summary = f"""
╔══════════════════════════════════════════════════════════════╗
║                    ИТОГОВЫЙ ОТЧЕТ                            ║
╠══════════════════════════════════════════════════════════════╣
║  Всего проанализировано компаний: {df_rfsd.shape[0]:>15,} ║
║  Всего кластеров координат:       {len(df_clusters):>15,} ║
║  Аномальных кластеров (>17):      {len(anomalous):>15,} ║
║  Доля аномалий:                   {len(anomalous)/len(df_clusters)*100:>14.1f}% ║
║  Максимум фирм на адрес:          {df_clusters['companies_count'].max():>15,} ║
╚══════════════════════════════════════════════════════════════╝

КЛЮЧЕВЫЕ НАХОДКИ:
1. {top10.iloc[0]['region']} — лидер по массовым регистрациям ({top10.iloc[0]['companies_count']} компаний)
2. {(anomalous['region'].value_counts().index[0] if len(anomalous) > 0 else 'N/A')} — регион с наибольшим числом аномальных кластеров
3. {(sizes > 100).sum()} кластеров имеют >100 компаний (критический риск)
"""

print(summary)


### Рекомендации для аудиторов и fraud-аналитиков:

1. **Приоритет проверки:** Сосредоточиться на ТОП-10 массовых адресов, особенно в Свердловской области и Краснодарском крае

2. **Автоматизация:** Использовать порог IQR (>17 компаний) для автоматической маркировки high-risk клиентов

3. **Мониторинг:** Настроить алерты на новые регистрации в уже выявленных аномальных кластерах

4. **Дальнейший анализ:** Расширить исследование анализом бенефициаров и связанных лиц (Graph Analysis)
